# SmartCrop — Notebook 07: Final Model

This notebook writes `models/trained/smartcrop_model.joblib` and `models/metadata/model_metadata.json`. The Django API loads those files and does not retrain them.


## 1. Introduction

One pipeline is saved: the algorithm chosen on validation R², fitted to the earlier seasons, with the feature builder and the median imputer inside the file. Sample predictions from the test seasons are exported with the report.


## 2. Objectives

1. Refit the selected model.
2. Record test metrics, row counts, features, seed, and library versions.
3. Save the joblib file, the metadata, sample predictions, and the HTML and PDF report.
4. Copy the artifact into `backend/ml_models/` for the API.


## 3. Dataset Description

Clean synthetic maize rows. Dataset version string: `synthetic-1.0.0`. Model version string: `1.0.0`.


## 4. Methodology

The selection rule is unchanged: highest validation R², then lower validation RMSE. Test metrics are stored after that choice. Metadata repeats the synthetic-data disclaimer so a later user of the file can see it without opening this notebook.


## 5. Implementation


In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    if (start / "data-science").is_dir():
        candidates.insert(0, start / "data-science")
    for candidate in candidates:
        if (candidate / "src" / "data" / "schema.py").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the SmartCrop repository.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 120)

import json
import shutil
from data.schema import SYNTHETIC_DISCLAIMER
from models.training import (
    RANDOM_SEED,
    build_metadata,
    evaluate_split,
    fit_final_model,
    permutation_feature_importance,
    run_model_comparison,
    save_model,
    select_model_name,
)
from reporting.model_report import prediction_frame, save_evaluation_figures, write_final_report

print(SYNTHETIC_DISCLAIMER)
clean = pd.read_csv(ROOT / "data" / "processed" / "clean_dataset.csv")
comparison, _details = run_model_comparison(clean, RANDOM_SEED)
chosen = select_model_name(comparison)
model, masks = fit_final_model(clean, chosen, RANDOM_SEED)
train = clean.loc[clean["year"].isin(masks["train_years"])]
validation = clean.loc[clean["year"].isin(masks["validation_years"])]
test = clean.loc[clean["year"].isin(masks["test_years"])]
validation_metrics = evaluate_split(model, validation)
test_metrics = evaluate_split(model, test)
importance = permutation_feature_importance(model, validation, RANDOM_SEED)
samples = prediction_frame(model, test)
metadata = build_metadata(
    chosen,
    comparison,
    test_metrics,
    validation_metrics,
    train_rows=len(train),
    validation_rows=len(validation),
    test_rows=len(test),
    seed=RANDOM_SEED,
    feature_importance=importance.round(6).to_dict(orient="records"),
)
paths = save_model(model, metadata, ROOT)
save_evaluation_figures(samples, importance, ROOT / "reports" / "figures")
reports = write_final_report(ROOT, metadata, comparison, importance, samples)
backend = ROOT.parent / "backend" / "ml_models"
backend.mkdir(parents=True, exist_ok=True)
shutil.copy2(paths["model"], backend / "smartcrop_model.joblib")
shutil.copy2(paths["metadata"], backend / "model_metadata.json")
print(json.dumps({
    "algorithm": metadata["algorithm"],
    "version": metadata["version"],
    "test_r2": metadata["metrics"]["test_r2"],
    "target_met": metadata["performance_target"]["met"],
    "model": str(paths["model"]),
    "html": str(reports["html"]),
    "pdf": str(reports["pdf"]),
}, indent=2))
display(samples.head(8))


SYNTHETIC DATA. These rows were generated by the SmartCrop synthetic generator for software and model-development exercises. They are not measurements from Macheke, Mashonaland East, Zimbabwe, and they must not be cited as agricultural statistics.


{
  "algorithm": "Random Forest Regressor",
  "version": "1.0.0",
  "test_r2": 0.4675,
  "target_met": false,
  "model": "/home/marvelous/.dev/Tait/capstone/data-science/models/trained/smartcrop_model.joblib",
  "html": "/home/marvelous/.dev/Tait/capstone/data-science/reports/final_model_report.html",
  "pdf": "/home/marvelous/.dev/Tait/capstone/data-science/reports/final_model_report.pdf"
}


,record_id,season,year,ward,yield_t_ha,predicted_yield_t_ha,residual_t_ha
703,SC-SYN-000723,2022/2023,2023,SYN-W01,2.099,2.076,0.023
704,SC-SYN-000734,2022/2023,2023,SYN-W01,1.988,2.036,-0.048
705,SC-SYN-000739,2022/2023,2023,SYN-W01,1.866,1.627,0.239
706,SC-SYN-000748,2022/2023,2023,SYN-W01,2.118,2.077,0.041
707,SC-SYN-000752,2022/2023,2023,SYN-W01,2.008,2.010,-0.002
708,SC-SYN-000765,2022/2023,2023,SYN-W01,1.828,1.938,-0.110
709,SC-SYN-000769,2022/2023,2023,SYN-W01,2.015,2.012,0.003
710,SC-SYN-000782,2022/2023,2023,SYN-W01,1.936,1.974,-0.038


## 6. Results

The printed JSON is the serialization receipt: algorithm, version, test R², whether the 0.80 R² target was met, and the file paths. Sample test predictions are displayed. They are model output, not observations.


## 7. Visualizations

The report embeds the actual-versus-predicted figure. The HTML and PDF reports are written to `reports/final_model_report.html` and `reports/final_model_report.pdf`.


## 8. Interpretation

Use the saved metrics as the performance of this synthetic experiment. A prediction from the API is an estimate from this pipeline. It is not a guaranteed yield, and it is not a measurement from a Macheke field.


## 9. Conclusion

The research path from the consolidated file to a versioned model file is complete for maize in the Macheke study area. Later seasons were held out. The API should load `smartcrop_model.joblib` and refuse to train during a request.
